In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import torch

print("GPU Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0))

GPU Available: True
GPU Name: Tesla T4


In [2]:
import os

print(os.listdir("/kaggle/input"))

[]


In [3]:
import os
print(os.listdir("/kaggle/input"))

['datasets']


In [4]:
import os

print(os.listdir("/kaggle/input/datasets"))

['olgaparfenova']


In [5]:
import os

print(os.listdir("/kaggle/input/datasets/olgaparfenova"))

['daisee']


In [6]:
import os

base_path = "/kaggle/input/datasets/olgaparfenova/daisee"
print(os.listdir(base_path))

['DAiSEE']


In [7]:
import os

base_path = "/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE"
print(os.listdir(base_path))

['Labels', 'DataSet', 'README.txt', 'GenderClips', 'hog.py', 'extractFrames.py']


In [8]:
video_path = "/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/DataSet"

for root, dirs, files in os.walk(video_path):
    print("Folder:", root)
    print("Sample files:", files[:5])
    break

Folder: /kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/DataSet
Sample files: ['Test.txt', 'Validation.txt', 'Train.txt']


In [9]:
video_path = "/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/DataSet"

for root, dirs, files in os.walk(video_path):
    print("Folder:", root)
    print("Sample video files:", files[:5])
    break

Folder: /kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/DataSet
Sample video files: ['Test.txt', 'Validation.txt', 'Train.txt']


In [11]:
import os
import cv2
import random

video_root = "/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/DataSet/Train"
output_root = "/kaggle/working/quick_data"

classes = ["Engaged", "Bored", "Confused", "Frustrated"]

# Create class folders
for c in classes:
    os.makedirs(os.path.join(output_root, c), exist_ok=True)

video_count = 0

for subject in os.listdir(video_root):
    subject_path = os.path.join(video_root, subject)
    
    for clip in os.listdir(subject_path):
        clip_path = os.path.join(subject_path, clip)
        
        for file in os.listdir(clip_path):
            if file.endswith(".mp4"):
                video_path = os.path.join(clip_path, file)
                
                cap = cv2.VideoCapture(video_path)
                saved = 0
                
                while saved < 3:
                    ret, frame = cap.read()
                    if not ret:
                        break
                    
                    class_name = random.choice(classes)  # fast demo
                    save_path = os.path.join(output_root, class_name, f"{file}_{saved}.jpg")
                    cv2.imwrite(save_path, frame)
                    
                    saved += 1
                
                cap.release()
                video_count += 1
                
                if video_count > 25:  # only 25 videos
                    break
        if video_count > 25:
            break
    if video_count > 25:
        break

print("Frames extracted from", video_count, "videos")

Frames extracted from 26 videos


In [12]:
for c in classes:
    print(c, ":", len(os.listdir(os.path.join(output_root, c))))

Engaged : 26
Bored : 18
Confused : 17
Frustrated : 17


In [13]:
!pip install timm

In [14]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import timm

device = torch.device("cuda")

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

dataset = datasets.ImageFolder(output_root, transform=transform)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

model = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True)
model.head = nn.Linear(model.head.in_features, 4)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

for epoch in range(2):
    correct = 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    print("Epoch:", epoch+1, "Accuracy:", 100*correct/total)

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

RuntimeError: only batches of spatial targets supported (3D tensors) but got targets of size: : [8]

In [15]:
print("Classes:", dataset.classes)
print("Number of classes:", len(dataset.classes))

Classes: ['Bored', 'Confused', 'Engaged', 'Frustrated']
Number of classes: 4


In [16]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import timm

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Dataset
dataset = datasets.ImageFolder(output_root, transform=transform)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

print("Classes:", dataset.classes)
print("Number of classes:", len(dataset.classes))
print("Total images:", len(dataset))

# Model
num_classes = len(dataset.classes)

model = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True)
model.head = nn.Linear(model.head.in_features, num_classes)
model = model.to(device)

# Loss & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

# Quick Training (2 epochs)
for epoch in range(2):
    model.train()
    correct = 0
    total = 0
    running_loss = 0
    
    for images, labels in loader:
        images = images.to(device)
        labels = labels.long().to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    print(f"Epoch {epoch+1} | Loss: {running_loss/len(loader):.4f} | Accuracy: {accuracy:.2f}%")

Using device: cuda
Classes: ['Bored', 'Confused', 'Engaged', 'Frustrated']
Number of classes: 4
Total images: 78


RuntimeError: only batches of spatial targets supported (3D tensors) but got targets of size: : [8]

In [17]:
import os

base_path = "/kaggle/working/quick_data"

for c in os.listdir(base_path):
    print(c, ":", len(os.listdir(os.path.join(base_path, c))))

Engaged : 26
Bored : 18
Frustrated : 17
Confused : 17


In [19]:

import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import timm

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Load dataset
dataset = datasets.ImageFolder("/kaggle/working/quick_data", transform=transform)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

print("Classes:", dataset.classes)
print("Total images:", len(dataset))

# Model
num_classes = len(dataset.classes)

model = timm.create_model(
    'swin_tiny_patch4_window7_224',
    pretrained=True,
    num_classes=num_classes   # IMPORTANT FIX
)

model = model.to(device)
# Loss & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

loader = DataLoader(dataset, batch_size=4, shuffle=True)

# Training
for epoch in range(20):
    model.train()
    correct = 0
    total = 0
    
    for images, labels in loader:
        images = images.to(device)
        labels = labels.long().to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    acc = 100 * correct / total
    print(f"Epoch {epoch+1} Accuracy: {acc:.2f}%")
    
    if acc > 90:
        print("Reached 90% accuracy. Stopping early.")
        break

Using device: cuda
Classes: ['Bored', 'Confused', 'Engaged', 'Frustrated']
Total images: 78
Epoch 1 Accuracy: 30.77%
Epoch 2 Accuracy: 23.08%
Epoch 3 Accuracy: 25.64%
Epoch 4 Accuracy: 32.05%
Epoch 5 Accuracy: 30.77%
Epoch 6 Accuracy: 26.92%
Epoch 7 Accuracy: 20.51%
Epoch 8 Accuracy: 24.36%
Epoch 9 Accuracy: 33.33%
Epoch 10 Accuracy: 33.33%
Epoch 11 Accuracy: 33.33%
Epoch 12 Accuracy: 29.49%
Epoch 13 Accuracy: 33.33%
Epoch 14 Accuracy: 33.33%
Epoch 15 Accuracy: 24.36%
Epoch 16 Accuracy: 33.33%
Epoch 17 Accuracy: 32.05%
Epoch 18 Accuracy: 33.33%
Epoch 19 Accuracy: 33.33%
Epoch 20 Accuracy: 30.77%


In [22]:
import pandas as pd

label_path = "/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/Labels/TrainLabels.csv"

df = pd.read_csv(label_path)

print(df.head())
print("Columns:", df.columns)
print("Total rows:", len(df))

           ClipID  Boredom  Engagement  Confusion  Frustration 
0  1100011002.avi        0           2          0             0
1  1100011003.avi        0           2          0             0
2  1100011004.avi        0           3          0             0
3  1100011005.avi        0           3          0             0
4  1100011006.avi        0           3          0             0
Columns: Index(['ClipID', 'Boredom', 'Engagement', 'Confusion', 'Frustration '], dtype='object')
Total rows: 5358


In [23]:
import os
import cv2
import pandas as pd

# Paths
label_path = "/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/Labels/TrainLabels.csv"
video_root = "/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/DataSet/Train"
output_root = "/kaggle/working/proper_data"

# Load CSV
df = pd.read_csv(label_path)

# Fix column name (remove trailing space)
df.columns = df.columns.str.strip()

# Engagement mapping
label_map = {
    0: "Bored",
    1: "Confused",
    2: "Engaged",
    3: "Frustrated"
}

# Create folders
for class_name in label_map.values():
    os.makedirs(os.path.join(output_root, class_name), exist_ok=True)

# Extract limited videos (for speed)
max_videos = 200
count = 0

for index, row in df.iterrows():
    
    clip_id = row["ClipID"]
    engagement = row["Engagement"]
    
    class_name = label_map[engagement]
    
    # Find video inside nested folders
    for subject in os.listdir(video_root):
        subject_path = os.path.join(video_root, subject)
        
        for clip_folder in os.listdir(subject_path):
            clip_path = os.path.join(subject_path, clip_folder)
            
            video_path = os.path.join(clip_path, clip_id)
            
            if os.path.exists(video_path):
                
                cap = cv2.VideoCapture(video_path)
                saved = 0
                
                while saved < 2:  # only 2 frames per video
                    ret, frame = cap.read()
                    if not ret:
                        break
                    
                    save_path = os.path.join(
                        output_root,
                        class_name,
                        f"{clip_id}_{saved}.jpg"
                    )
                    
                    cv2.imwrite(save_path, frame)
                    saved += 1
                
                cap.release()
                count += 1
                break
        
        if count >= max_videos:
            break
    
    if count >= max_videos:
        break

print("Total videos processed:", count)

Total videos processed: 200


In [24]:
for c in os.listdir(output_root):
    print(c, ":", len(os.listdir(os.path.join(output_root, c))))

Engaged : 120
Bored : 0
Frustrated : 266
Confused : 14


In [26]:
import os

video_root = "/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/DataSet/Train"

video_dict = {}

for root, dirs, files in os.walk(video_root):
    for file in files:
        if file.endswith(".avi"):
            video_dict[file] = os.path.join(root, file)

print("Total videos found:", len(video_dict))

Total videos found: 4976


In [27]:
import os
import cv2
import pandas as pd

# Paths
label_path = "/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/Labels/TrainLabels.csv"
output_root = "/kaggle/working/proper_data"

# Load labels
df = pd.read_csv(label_path)
df.columns = df.columns.str.strip()

# Engagement mapping
label_map = {
    0: "Bored",
    1: "Confused",
    2: "Engaged",
    3: "Frustrated"
}

# Create class folders
for class_name in label_map.values():
    os.makedirs(os.path.join(output_root, class_name), exist_ok=True)

max_videos = 200   # keep small for speed
count = 0

for index, row in df.iterrows():
    
    clip_id = row["ClipID"]
    engagement = row["Engagement"]
    
    if clip_id in video_dict:
        
        class_name = label_map[engagement]
        video_path = video_dict[clip_id]
        
        cap = cv2.VideoCapture(video_path)
        saved = 0
        
        while saved < 2:   # 2 frames per video
            ret, frame = cap.read()
            if not ret:
                break
            
            save_path = os.path.join(
                output_root,
                class_name,
                f"{clip_id}_{saved}.jpg"
            )
            
            cv2.imwrite(save_path, frame)
            saved += 1
        
        cap.release()
        count += 1
    
    if count >= max_videos:
        break

print("Total videos processed:", count)

Total videos processed: 200


In [28]:
import os

for c in os.listdir(output_root):
    print(c, ":", len(os.listdir(os.path.join(output_root, c))))

Engaged : 120
Bored : 0
Frustrated : 266
Confused : 14


In [29]:
import shutil

shutil.rmtree("/kaggle/working/proper_data", ignore_errors=True)
print("Old dataset cleared")

Old dataset cleared


In [30]:
import os
import cv2
import pandas as pd

label_path = "/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/Labels/TrainLabels.csv"
output_root = "/kaggle/working/proper_data"

df = pd.read_csv(label_path)
df.columns = df.columns.str.strip()

label_map = {
    0: "Bored",
    1: "Confused",
    2: "Engaged",
    3: "Frustrated"
}

# Create folders
for class_name in label_map.values():
    os.makedirs(os.path.join(output_root, class_name), exist_ok=True)

# Track how many per class
target_per_class = 80
class_counts = {c: 0 for c in label_map.values()}

for index, row in df.iterrows():
    
    engagement = row["Engagement"]
    clip_id = row["ClipID"]
    
    class_name = label_map[engagement]
    
    if class_counts[class_name] >= target_per_class:
        continue
    
    if clip_id in video_dict:
        
        video_path = video_dict[clip_id]
        cap = cv2.VideoCapture(video_path)
        saved = 0
        
        while saved < 2:
            ret, frame = cap.read()
            if not ret:
                break
            
            save_path = os.path.join(
                output_root,
                class_name,
                f"{clip_id}_{saved}.jpg"
            )
            
            cv2.imwrite(save_path, frame)
            saved += 1
        
        cap.release()
        class_counts[class_name] += 1
    
    # Stop if all classes filled
    if all(count >= target_per_class for count in class_counts.values()):
        break

print("Videos per class:", class_counts)

Videos per class: {'Bored': 33, 'Confused': 80, 'Engaged': 80, 'Frustrated': 80}


In [31]:
!pip install timm -q

import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import timm
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Transform
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

# Load dataset
data_path = "/kaggle/working/proper_data"
dataset = datasets.ImageFolder(data_path, transform=transform)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

print("Classes:", dataset.classes)
print("Total images:", len(dataset))

# Model
model = timm.create_model(
    'swin_tiny_patch4_window7_224',
    pretrained=True,
    num_classes=len(dataset.classes)
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)

# Train
for epoch in range(8):   # 8 epochs for better learning
    model.train()
    correct = 0
    total = 0
    
    for images, labels in loader:
        images = images.to(device)
        labels = labels.long().to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    acc = 100 * correct / total
    print(f"Epoch {epoch+1} Accuracy: {acc:.2f}%")

Using device: cuda
Classes: ['Bored', 'Confused', 'Engaged', 'Frustrated']
Total images: 546
Epoch 1 Accuracy: 26.37%
Epoch 2 Accuracy: 27.66%
Epoch 3 Accuracy: 30.04%
Epoch 4 Accuracy: 30.95%
Epoch 5 Accuracy: 29.30%
Epoch 6 Accuracy: 25.46%
Epoch 7 Accuracy: 25.82%
Epoch 8 Accuracy: 27.84%


In [32]:
!pip install timm -q

import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import timm
import os

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Better Transform (adds slight augmentation)
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

# Load dataset
data_path = "/kaggle/working/proper_data"
dataset = datasets.ImageFolder(data_path, transform=transform)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

print("Classes:", dataset.classes)
print("Total images:", len(dataset))

# Create model
model = timm.create_model(
    'swin_tiny_patch4_window7_224',
    pretrained=True,
    num_classes=len(dataset.classes)
)

# 🔥 Freeze entire backbone
for param in model.parameters():
    param.requires_grad = False

# 🔥 Unfreeze only classification head
for param in model.head.parameters():
    param.requires_grad = True

model = model.to(device)

# Loss and Optimizer (train head only)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.head.parameters(), lr=0.001)

# Training
epochs = 12

for epoch in range(epochs):
    model.train()
    correct = 0
    total = 0
    running_loss = 0
    
    for images, labels in loader:
        images = images.to(device)
        labels = labels.long().to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    acc = 100 * correct / total
    print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(loader):.4f} | Accuracy: {acc:.2f}%")

Using device: cuda
Classes: ['Bored', 'Confused', 'Engaged', 'Frustrated']
Total images: 546
Epoch 1/12 | Loss: 0.9043 | Accuracy: 67.95%
Epoch 2/12 | Loss: 0.6437 | Accuracy: 79.85%
Epoch 3/12 | Loss: 0.5634 | Accuracy: 83.88%
Epoch 4/12 | Loss: 0.5238 | Accuracy: 83.52%
Epoch 5/12 | Loss: 0.4975 | Accuracy: 84.43%
Epoch 6/12 | Loss: 0.4568 | Accuracy: 84.07%
Epoch 7/12 | Loss: 0.4558 | Accuracy: 84.80%
Epoch 8/12 | Loss: 0.4356 | Accuracy: 86.45%
Epoch 9/12 | Loss: 0.3965 | Accuracy: 86.63%
Epoch 10/12 | Loss: 0.3980 | Accuracy: 85.71%
Epoch 11/12 | Loss: 0.3822 | Accuracy: 86.81%
Epoch 12/12 | Loss: 0.3701 | Accuracy: 86.63%
